# Predict New York City Airbnb Prices
**AIML Minor Project 2 — Linear Regression**

**Objective:** Build a Linear Regression model to estimate the nightly price of Airbnb listings in New York City using the AB_NYC_2019 dataset.

**Dataset:** New York City Airbnb Open Data (2019), commonly containing 48,895 rows and 16 columns.

> Run the notebook from top to bottom. If the dataset is not already available in the current folder, the first code cell will download the public CSV. If that download is unavailable, upload `AB_NYC_2019.csv` when prompted.

## 1. Import libraries and load the dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Dataset source: public copy of the AB_NYC_2019 dataset.
DATA_URL = "https://raw.githubusercontent.com/YueminLi/Airbnb_NYC_2019/master/AB_NYC_2019.csv"
DATA_FILE = "AB_NYC_2019.csv"

if os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE)
else:
    try:
        df = pd.read_csv(DATA_URL)
        df.to_csv(DATA_FILE, index=False)
    except Exception:
        try:
            from google.colab import files
            print("Automatic download was unavailable. Please upload AB_NYC_2019.csv")
            uploaded = files.upload()
            uploaded_name = next(iter(uploaded))
            df = pd.read_csv(uploaded_name)
        except Exception as e:
            raise RuntimeError("Please place AB_NYC_2019.csv in the notebook folder and run again.") from e

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 2. Exploratory Data Analysis (EDA) and data auditing

In [ ]:
# Task 1.1: Inspect shape, columns and data types
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

In [ ]:
# Task 1.2: Classify variables
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

# Discrete and continuous numerical variables (based on their meaning)
discrete_numeric = [
    "id", "host_id", "price", "minimum_nights", "number_of_reviews",
    "calculated_host_listings_count", "availability_365"
]
continuous_numeric = ["latitude", "longitude", "reviews_per_month"]

print("\nDiscrete numerical:", [c for c in discrete_numeric if c in df.columns])
print("Continuous numerical:", [c for c in continuous_numeric if c in df.columns])
print("Nominal categorical: neighbourhood_group, neighbourhood, room_type, names")
print("Ordinal categorical: No clear ordinal feature is used in this dataset.")

In [ ]:
# Task 1.3: Missing-value count and percentage
missing_audit = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})
display(missing_audit.sort_values("Missing_Count", ascending=False))

## 3. Data cleaning and handling missing values

In [ ]:
clean = df.copy()

# Task 2.1: last_review has many missing date values and is not required
# for this basic price model, so the column is dropped.
if "last_review" in clean.columns:
    clean = clean.drop(columns=["last_review"])

# Task 2.2: Missing reviews_per_month generally correspond to listings
# with zero reviews. Fill those rows with 0; use the median for any
# remaining missing values.
if "reviews_per_month" in clean.columns:
    no_reviews = clean["number_of_reviews"].fillna(0).eq(0)
    clean.loc[no_reviews & clean["reviews_per_month"].isna(), "reviews_per_month"] = 0
    clean["reviews_per_month"] = clean["reviews_per_month"].fillna(
        clean["reviews_per_month"].median()
    )

# Remove rows where the target is missing or non-positive.
clean = clean.dropna(subset=["price"])
clean = clean[clean["price"] > 0].copy()

# Task 2.3: Drop identifiers, names and host names that do not provide
# useful general numeric information for this model.
identifier_columns = ["id", "name", "host_id", "host_name"]
clean = clean.drop(columns=[c for c in identifier_columns if c in clean.columns])

print("Shape after basic cleaning:", clean.shape)
print("Remaining missing values:")
display(clean.isna().sum().to_frame("Missing_Count"))

## 4. Outlier identification and treatment

In [ ]:
# Task 3.1: Visualize price and minimum_nights before treatment
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(x=clean["price"], ax=axes[0])
axes[0].set_title("Price — before outlier treatment")
sns.boxplot(x=clean["minimum_nights"], ax=axes[1])
axes[1].set_title("Minimum nights — before outlier treatment")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(clean["price"], bins=50, ax=axes[0])
axes[0].set_title("Price distribution")
sns.histplot(clean["minimum_nights"], bins=50, ax=axes[1])
axes[1].set_title("Minimum nights distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Task 3.2: Trim very high prices using the 95th percentile.
# Also remove listings with minimum stays of 365 nights or more.
price_95 = clean["price"].quantile(0.95)
before_rows = len(clean)
clean = clean[(clean["price"] <= price_95) & (clean["minimum_nights"] < 365)].copy()

print(f"95th percentile price threshold: ${price_95:.2f}")
print("Rows before trimming:", before_rows)
print("Rows after trimming:", len(clean))
print("Rows removed:", before_rows - len(clean))

## 5. Feature selection and categorical encoding

In [ ]:
# Task 4.1: Drop neighbourhood because it has 200+ distinct values.
# Keeping it would create many extra columns and make this simple model
# unnecessarily complex (high cardinality / curse of dimensionality).
if "neighbourhood" in clean.columns:
    print("Unique neighbourhoods:", clean["neighbourhood"].nunique())
    clean = clean.drop(columns=["neighbourhood"])

# Task 4.2: One-Hot Encode the two categorical features and drop the first
# category from each feature to avoid the dummy variable trap.
categorical_features = ["neighbourhood_group", "room_type"]
clean = pd.get_dummies(
    clean,
    columns=[c for c in categorical_features if c in clean.columns],
    drop_first=True,
    dtype=int
)

print("Encoded dataset shape:", clean.shape)
display(clean.head())

## 6. Feature-target separation and train-test split

In [ ]:
# Task 5.1: Separate features (X) and target (y).
X = clean.drop(columns=["price"])
y = clean["price"]

# Keep only numeric columns for Linear Regression.
X = X.select_dtypes(include=np.number).copy()
X = X.fillna(X.median())

# Task 5.2: 80% training and 20% testing with a fixed seed.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

## 7. Model training and evaluation

In [ ]:
# Task 6.1: Initialize and train the Linear Regression model.
model = LinearRegression()
model.fit(X_train, y_train)

# Task 6.2: Predict prices for the test set.
y_pred = model.predict(X_test)

# Task 6.3: Calculate evaluation metrics.
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R-squared (R²): {r2:.4f}")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")

## 8. Residual analysis and interpretation

In [ ]:
# Task 7.1: Actual prices vs predicted prices with a 45-degree reference line.
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.35)
lower = min(y_test.min(), y_pred.min())
upper = max(y_test.max(), y_pred.max())
plt.plot([lower, upper], [lower, upper], linestyle="--")
plt.xlabel("Actual Price (USD)")
plt.ylabel("Predicted Price (USD)")
plt.title("Actual vs Predicted Airbnb Prices")
plt.tight_layout()
plt.show()

In [ ]:
# Task 7.2: Residual plot and residual distribution.
residuals = y_test - y_pred

plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price (USD)")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.histplot(residuals, bins=40, kde=True)
plt.xlabel("Residual (USD)")
plt.title("Distribution of Prediction Errors")
plt.tight_layout()
plt.show()

In [ ]:
# Task 7.3: Compare errors for cheaper and more expensive test listings.
# The test data is split at its median actual price.
comparison = pd.DataFrame({
    "Actual_Price": y_test,
    "Predicted_Price": y_pred
})
comparison["Absolute_Error"] = (
    comparison["Actual_Price"] - comparison["Predicted_Price"]
).abs()
median_test_price = comparison["Actual_Price"].median()
comparison["Price_Group"] = np.where(
    comparison["Actual_Price"] <= median_test_price,
    "Lower-priced (at/below median)",
    "Higher-priced (above median)"
)

group_errors = comparison.groupby("Price_Group")["Absolute_Error"].agg(
    ["count", "mean", "median"]
)
display(group_errors)

lower_mae = group_errors.loc["Lower-priced (at/below median)", "mean"]
higher_mae = group_errors.loc["Higher-priced (above median)", "mean"]

if lower_mae < higher_mae:
    interpretation = "The model has a lower average absolute error on lower-priced listings."
elif higher_mae < lower_mae:
    interpretation = "The model has a lower average absolute error on higher-priced listings."
else:
    interpretation = "The average absolute errors are equal in both groups."

print(interpretation)
print("This comparison describes this test split; it does not prove that price alone causes the difference.")

## 9. Conclusion

This project uses the New York City Airbnb Open Data to estimate listing prices with Linear Regression. The dataset is inspected, cleaned, and prepared by handling missing values, removing extreme observations, selecting useful features, and encoding categorical variables. The model is trained on 80% of the data and evaluated on the remaining 20%.

The R² score describes how much variation in test prices is explained by the model. MAE gives the average absolute prediction error in US dollars, while MSE and RMSE give greater weight to larger errors. The actual values printed by the notebook should be copied into the one-page summary report.

**Limitation:** Airbnb prices can depend on many factors not fully represented by this simple linear model. Results are estimates, not guaranteed market prices.